# 05 — CLIP Clustering V2 : UMAP + HDBSCAN

V1 (PCA 50D + KMeans k=5) abandonnée — clusters fourre-tout, doublons de labels, k fixé arbitrairement.

V2 : UMAP (réduction non-linéaire, métrique cosine) + HDBSCAN (clustering densité, k automatique, outliers = -1).

- Input  : data/warehouse/photo_embeddings/
- Output : data/warehouse/photo_clusters/

In [1]:
import sys, os
from pathlib import Path

_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)
from config import WAREHOUSE, CLIP_CANDIDATE_LABELS

EMBEDDINGS_DIR = Path(WAREHOUSE) / 'photo_embeddings'
CLUSTERS_DIR   = Path(WAREHOUSE) / 'photo_clusters'

CLIP_MODEL = 'openai/clip-vit-large-patch14'
# CANDIDATE_LABELS lu depuis config.yaml -> config.py -> CLIP_CANDIDATE_LABELS
CANDIDATE_LABELS = CLIP_CANDIDATE_LABELS

print(f'Embeddings : {EMBEDDINGS_DIR}')
print(f'Clusters   : {CLUSTERS_DIR}')
print(f'Labels     : {len(CANDIDATE_LABELS)} categories')


Embeddings : /opt/spark/data/warehouse/photo_embeddings
Clusters   : /opt/spark/data/warehouse/photo_clusters
Labels     : 13 categories


## 1. Lecture des embeddings via Spark

In [2]:
from config import build_spark_session
from delta.tables import DeltaTable

spark = build_spark_session("MyDigitalTwin-CLIP-Clustering-V2", delta=True)
spark.sparkContext.setLogLevel('WARN')

embeddings_path = str(EMBEDDINGS_DIR)
if DeltaTable.isDeltaTable(spark, embeddings_path):
    df = spark.read.format("delta").load(embeddings_path).cache()
else:
    df = spark.read.parquet(embeddings_path).cache()

print(f'Photos chargées : {df.count()}')
df.printSchema()

26/05/12 18:04:07 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
26/05/12 18:04:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Photos chargées : 2420
root
 |-- path: string (nullable = true)
 |-- filename: string (nullable = true)
 |-- embedding: array (nullable = true)
 |    |-- element: float (containsNull = true)



## 2. Collect -> numpy

Pour 2 419 photos, un seul `.collect()` est acceptable. UMAP et HDBSCAN ne sont pas dans MLlib.

In [3]:
import numpy as np

print('Collecte des embeddings...')
rows = df.select('path', 'filename', 'embedding').collect()

paths     = [r['path']     for r in rows]
filenames = [r['filename'] for r in rows]
X = np.array([r['embedding'] for r in rows], dtype=np.float32)

print(f'Shape embeddings : {X.shape}')
print(f'Norme L2 moyenne : {np.linalg.norm(X, axis=1).mean():.4f}  (attendu ~1.0)')

Collecte des embeddings...
Shape embeddings : (2420, 768)
Norme L2 moyenne : 1.0000  (attendu ~1.0)


## 3. UMAP — 768D -> 50D (clustering) + 2D (visualisation)

In [4]:
import umap

print('1/2 - UMAP 768D -> 50D (clustering)...')
reducer_50 = umap.UMAP(
    # n_components : dimension de l'espace réduit pour HDBSCAN.
    # 768D → 50D : réduit le bruit dimensionnel sans écraser la structure locale.
    # Trop bas (ex : 2D) = perte d'information pour le clustering.
    # Trop haut (ex : 200D) = malédiction de la dimensionnalité pour HDBSCAN.
    n_components=50,

    # n_neighbors : taille du voisinage local pour construire le graphe de similarité.
    # Petit (ex : 5) = structure fine et locale, sensible au bruit.
    # Grand (ex : 50) = structure globale, clusters plus larges.
    # 15 est le défaut UMAP — bon équilibre sur ~2400 points.
    n_neighbors=15,

    # min_dist : distance minimale entre deux points dans l'espace réduit.
    # 0.0 = points compressés en clusters très denses.
    # 1.0 = distribution quasi-uniforme, structure locale perdue.
    # 0.1 : préserve la structure locale sans sur-comprimer.
    min_dist=0.1,

    # metric : distance utilisée dans l'espace original (768D).
    # Embeddings CLIP L2-normalisés → cosine distance = distance euclidienne sur l'hypersphère.
    # Utiliser 'cosine' explicitement est plus correct que 'euclidean' ici.
    metric='cosine',

    random_state=42,
    verbose=True,
)
X_50 = reducer_50.fit_transform(X)
print(f'Shape après UMAP 50D : {X_50.shape}')

print('2/2 - UMAP 768D -> 2D (visualisation dashboard)...')
reducer_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42,
)
X_2d = reducer_2d.fit_transform(X)
print(f'Shape UMAP 2D : {X_2d.shape}')

1/2 - UMAP 768D -> 50D (clustering)...
UMAP(angular_rp_forest=True, metric='cosine', n_components=50, n_jobs=1, random_state=42, verbose=True)
Tue May 12 18:04:39 2026 Construct fuzzy simplicial set


/usr/local/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Tue May 12 18:04:44 2026 Finding Nearest Neighbors
Tue May 12 18:04:46 2026 Finished Nearest Neighbor Search
Tue May 12 18:04:48 2026 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Tue May 12 18:04:53 2026 Finished embedding
Shape après UMAP 50D : (2420, 50)
2/2 - UMAP 768D -> 2D (visualisation dashboard)...


/usr/local/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Shape UMAP 2D : (2420, 2)


## 4. HDBSCAN — clustering densite (k automatique)

In [5]:
import hdbscan
import pandas as pd

print('HDBSCAN...')
clustering = hdbscan.HDBSCAN(
    # min_cluster_size : taille minimale d'un cluster.
    # Sur 2419 photos, 50 ≈ 2% du corpus.
    # Trop petit (ex : 10) → micro-clusters parasites (testé avec 30 : trop fragmenté).
    # Trop grand (ex : 200) → clusters fusionnés, perte de granularité.
    min_cluster_size=50,

    # min_samples : nombre de voisins requis pour qu'un point soit "core point".
    # Petit (ex : 1) = moins de bruit, clusters plus inclusifs.
    # Grand (ex : 20) = clusters plus denses, plus de points classés bruit (-1).
    # 5 : bruit modéré — les photos atypiques vont en bruit sans forcer leur assignation.
    min_samples=5,

    # metric : distance utilisée sur l'espace UMAP 50D.
    # Ici 'euclidean' est correct : UMAP produit des coordonnées cartésiennes,
    # pas des embeddings normalisés. Différent de la métrique cosine utilisée dans UMAP.
    metric='euclidean',

    # cluster_selection_method : stratégie de sélection des clusters dans la hiérarchie.
    # 'eom' (Excess of Mass) : favorise les clusters stables de tailles inégales — réaliste
    # ici (soirées = 1295 photos vs enfance = 67 photos).
    # Alternatif 'leaf' : clusters plus petits et équilibrés, moins adapté à ce corpus.
    cluster_selection_method='eom',
)
labels = clustering.fit_predict(X_50)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = (labels == -1).sum()

print(f'Clusters trouvés : {n_clusters}')
print(f'Outliers (bruit) : {n_noise} photos ({n_noise/len(labels)*100:.1f}%)')

print('\nDistribution :')
series = pd.Series(labels)
print(series.value_counts().sort_index().rename(index={-1: 'bruit (-1)'}).to_string())

HDBSCAN...
Clusters trouvés : 5
Outliers (bruit) : 288 photos (11.9%)

Distribution :
bruit (-1)     288
0               67
1               83
2              252
3               65
4             1665


## 5. Labels manuels (basés sur inspection visuelle des clusters)

In [6]:
cluster_labels = {
  -1: "Souvenir en tout genre",
   0: "photos d'enfance",
   1: "Memes et Humour",
   2: "MDR ya que des meufs",
   3: "Voyage Rhéto",
   4: "Soirées"
}

print("Labels :")
for cid, label in sorted(cluster_labels.items()):
  mask = labels == cid
  print(f"  Cluster {cid:>2} ({mask.sum():>4} photos) -> {label}")

Labels :
  Cluster -1 ( 288 photos) -> Souvenir en tout genre
  Cluster  0 (  67 photos) -> photos d'enfance
  Cluster  1 (  83 photos) -> Memes et Humour
  Cluster  2 ( 252 photos) -> MDR ya que des meufs
  Cluster  3 (  65 photos) -> Voyage Rhéto
  Cluster  4 (1665 photos) -> Soirées


## 6. Sauvegarde via Spark

In [7]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from delta.tables import DeltaTable

rows_out = [
    (
        paths[i],
        filenames[i],
        int(labels[i]),
        cluster_labels[int(labels[i])],
        float(X_2d[i, 0]),
        float(X_2d[i, 1]),
    )
    for i in range(len(paths))
]

schema = StructType([
    StructField('path',          StringType(),  nullable=False),
    StructField('filename',      StringType(),  nullable=False),
    StructField('cluster',       IntegerType(), nullable=False),
    StructField('cluster_label', StringType(),  nullable=False),
    StructField('umap_x',        FloatType(),   nullable=False),
    StructField('umap_y',        FloatType(),   nullable=False),
])

df_out = spark.createDataFrame(rows_out, schema=schema)

# ── Stratégie : delete + append ───────────────────────────────────────────────
# Type : table calculée — UMAP + HDBSCAN entièrement recalculés à chaque run.
# Les labels peuvent changer entre runs → MERGE laisserait des lignes orphelines.
clusters_path = str(CLUSTERS_DIR)
CLUSTERS_DIR.mkdir(parents=True, exist_ok=True)
if DeltaTable.isDeltaTable(spark, clusters_path):
    DeltaTable.forPath(spark, clusters_path).delete()
    df_out.write.format("delta").mode("append").save(clusters_path)
else:
    df_out.write.format("delta").mode("overwrite").save(clusters_path)

print(f'Clusters sauvegardes -> {CLUSTERS_DIR}')
spark.read.format("delta").load(clusters_path).groupBy('cluster', 'cluster_label').count().orderBy('cluster').show(truncate=False)

Clusters sauvegardes -> /opt/spark/data/warehouse/photo_clusters


+-------+----------------------+-----+
|cluster|cluster_label         |count|
+-------+----------------------+-----+
|-1     |Souvenir en tout genre|288  |
|0      |photos d'enfance      |67   |
|1      |Memes et Humour       |83   |
|2      |MDR ya que des meufs  |252  |
|3      |Voyage Rhéto          |65   |
|4      |Soirées               |1665 |
+-------+----------------------+-----+



In [8]:
spark.stop()